In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Inicjalizacja Qubitów

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie powstał przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych wersji lub nowszych.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Gdy Circuit jest wykonywany na jednostce przetwarzania kwantowego (QPU) IBM&reg;, na początku Circuit zazwyczaj wstawiany jest niejawny reset, który zapewnia inicjalizację Qubitów do zera. Jest to kontrolowane przez flagę `init_qubits`, ustawianą jako [opcja wykonania prymitywu](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Jednak proces resetu nie jest doskonały, co prowadzi do błędów przygotowania stanu. Aby złagodzić ten błąd, system wstawia też czas opóźnienia powtórzeń (ang. `rep_delay`) pomiędzy Circuit. Każdy Backend ma inną domyślną wartość `rep_delay`, ale zwykle jest ona dłuższa niż T1, aby umożliwić środowisku zresetowanie Qubitów. Domyślną wartość `rep_delay` można sprawdzić, wywołując `backend.default_rep_delay`.

Wszystkie QPU IBM korzystają z wykonywania z dynamiczną częstotliwością powtórzeń, co pozwala zmieniać `rep_delay` dla każdego zadania. Circuit przesłane w ramach zadania prymitywu są grupowane w paczki do wykonania na QPU. Te Circuit są wykonywane przez iterację po Circuit dla każdego żądanego strzału (ang. shot); wykonanie przebiega kolumnowo po macierzy Circuit i strzałów, jak pokazano na poniższym rysunku.

![Pierwsza kolumna reprezentuje strzał 0. Circuit są uruchamiane kolejno od 0 do 3. Druga kolumna reprezentuje strzał 1. Circuit są uruchamiane kolejno od 0 do 3. Pozostałe kolumny podążają za tym samym schematem.](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Macierz wykonania kolumnowego")

Ponieważ `rep_delay` jest wstawiany pomiędzy Circuit, każdy strzał podczas wykonania napotyka to opóźnienie. Dlatego obniżenie wartości `rep_delay` skraca całkowity czas wykonania na QPU, ale kosztem wzrostu wskaźnika błędów przygotowania stanu, co widać na poniższym obrazie:

![Ten obraz pokazuje, że wraz ze zmniejszaniem wartości `rep_delay` wskaźnik błędów przygotowania stanu rośnie.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Opóźnienie powtórzeń a wskaźnik błędów")

Ustawienie zarówno `rep_delay=0`, jak i `init_qubits=False` w praktyce „scala" Circuit, ponieważ Qubity będą zaczynać w stanie końcowym z poprzedniego strzału.

Pamiętaj, że choć Circuit w ramach zadania prymitywu są grupowane do wykonania na QPU, kolejność wykonywania Circuit z poszczególnych PUBów nie jest gwarantowana. Dlatego nawet jeśli przesyłasz `pubs=[pub1, pub2]`, nie ma gwarancji, że Circuit z `pub1` zostaną uruchomione przed Circuit z `pub2`. Nie ma też gwarancji, że Circuit z tego samego zadania zostaną uruchomione jako jedna paczka na QPU.

## Określanie rep_delay dla zadania prymitywu

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Następne kroki
> **Tip:** - Wypróbuj przykład z samouczka [Kwantowy algorytm aproksymacyjnej optymalizacji (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Zapoznaj się z tym, jak [zacząć pracę z prymitywami.](./get-started-with-primitives)